# Question Predictor — 10-Year Paper Data

In [1]:
# Install dependencies (run once)
# !pip install pandas scikit-learn transformers torch datasets

## 1. Load Data

In [30]:
import pandas as pd

# Expected CSV columns: 'year', 'subject', 'topic', 'question'
df = pd.read_csv('data/training-dataset.csv')
print(df.shape)
df.head()

(1739, 8)


,year,sample_question_id,normalized_question,question_text,topic,source_file,occurrence_count,first_line
0,4th Year,618e82323171,attempt any two parts of the following,Attempt any TWO parts of the following,(C),4th-year.pdf,3,1188
1,4th Year,f7c1380a0d71,explain,Explain.,"EVEN SEMESTER EXAMINATION, 2023 – 24",4th-year.pdf,3,1854
2,4th Year,534e8c894772,t o,T.O.,P.T.O.,4th-year.pdf,3,3001
3,4th Year,c95ee47689a0,tech,TECH. (SEMESTER-VII),B.TECH. (SEMESTER-VII),4th-year.pdf,3,5611
4,4th Year,ef44c3d7b52f,a explain web service architecture,a) Explain Web Service Architecture?,Q3.,4th-year.pdf,2,2595


## 2. Preprocess

In [23]:
# Preprocess and split data for model training
# Use available columns: topic, year, normalized_question for input; question_text as target

df['input_text'] = df['topic'].astype(str) + ' | ' + df['year'].astype(str) + ' | ' + df['normalized_question'].astype(str)
df = df.dropna(subset=['input_text', 'question_text'])
# Manual split to avoid issues with train_test_split
train_df, test_df = df.iloc[:int(0.8*len(df))], df.iloc[int(0.8*len(df)):]  # 80/20 split
print(f'Train: {len(train_df)} | Test: {len(test_df)}')

Train: 1391 | Test: 348


## 3. Tokenize

In [24]:
from transformers import T5Tokenizer

tokenizer = T5Tokenizer.from_pretrained('t5-small')

def tokenize(texts, targets, max_len=128):
    inputs = tokenizer(texts, max_length=max_len, truncation=True, padding='max_length', return_tensors='pt')
    labels = tokenizer(targets, max_length=max_len, truncation=True, padding='max_length', return_tensors='pt').input_ids
    return inputs, labels

train_inputs, train_labels = tokenize(train_df['input_text'].tolist(), train_df['question_text'].tolist())
test_inputs,  test_labels  = tokenize(test_df['input_text'].tolist(),  test_df['question_text'].tolist())

## 4. Dataset & DataLoader

In [25]:
import torch
from torch.utils.data import Dataset, DataLoader

class PaperDataset(Dataset):
    def __init__(self, inputs, labels):
        self.inputs = inputs
        self.labels = labels

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, idx):
        return {
            'input_ids':      self.inputs.input_ids[idx],
            'attention_mask': self.inputs.attention_mask[idx],
            'labels':         self.labels[idx]
        }

train_loader = DataLoader(PaperDataset(train_inputs, train_labels), batch_size=8, shuffle=True)
test_loader  = DataLoader(PaperDataset(test_inputs,  test_labels),  batch_size=8)

## 5. Model

In [26]:
from transformers import T5ForConditionalGeneration

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = T5ForConditionalGeneration.from_pretrained('t5-small').to(device)
print(f'Using device: {device}')

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 891.15it/s]


Using device: cpu


## 7. Predict Questions & Extract Topics for 3rd Year

In [28]:
# Load pre-trained model from saved folder
model = T5ForConditionalGeneration.from_pretrained('saved_model/').to(device)
tokenizer = T5Tokenizer.from_pretrained('saved_model/')
print('Pre-trained model loaded successfully!')

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 1596.61it/s]


Pre-trained model loaded successfully!


In [ ]:
## 6. Load Pre-trained Model (Skip Training)

KeyboardInterrupt: 

## 7. Evaluate & Predict

In [31]:
model.eval()

def predict(topic, year, normalized_question):
    text = f'{topic} | {year} | {normalized_question}'
    inputs = tokenizer(text, return_tensors='pt').to(device)
    output = model.generate(**inputs, max_length=128)
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Check available years in the dataset
print("Available years in dataset:")
print(df['year'].unique())
print(f"\nYear value counts:\n{df['year'].value_counts()}")

# Get 3rd year data (may be stored as "3rd Year" or similar)
third_year_data = df[df['year'].str.contains('3', case=False, na=False)]
if len(third_year_data) == 0:
    # If no 3rd year, use 4th year as example (which we have)
    print("\n⚠️  No 3rd year data found. Using available data to extract important topics...")
    year_to_analyze = df['year'].unique()[0]
    third_year_data = df[df['year'] == year_to_analyze]
else:
    year_to_analyze = "3rd Year"

print(f"\nTotal questions analyzed: {len(third_year_data)}")
print(f"Year: {year_to_analyze}")
print("\n=== IMPORTANT TOPICS FOR STUDENTS ===\n")

# Get unique topics and their frequency
topic_counts = third_year_data['topic'].value_counts()
print(f"Number of unique topics: {len(topic_counts)}")
print("\nTop Topics by Frequency:")
for topic, count in topic_counts.head(15).items():
    percentage = (count / len(third_year_data)) * 100
    print(f"  • {topic}: {count} questions ({percentage:.1f}%)")

Available years in dataset:
<StringArray>
['4th Year', '3rd Year']
Length: 2, dtype: str

Year value counts:
year
4th Year    1083
3rd Year     656
Name: count, dtype: int64

Total questions analyzed: 656
Year: 3rd Year

=== IMPORTANT TOPICS FOR STUDENTS ===

Number of unique topics: 120

Top Topics by Frequency:
  • [P.T.O.]: 40 questions (6.1%)
  • DATA ANALYTICS: 34 questions (5.2%)
  • ARTIFICIAL INTELLIGENCE: 24 questions (3.7%)
  • COMPUTER NETWORKS: 22 questions (3.4%)
  • Q.4.: 20 questions (3.0%)
  • EVEN SEMESTER EXAMINATION, 2023 – 24: 15 questions (2.3%)
  • TCS-604 // 2000: 14 questions (2.1%)
  • TCS-402/1960/4: 14 questions (2.1%)
  • TCS-601/1970/4: 14 questions (2.1%)
  • ANALYSIS OF ALGORITHMS: 13 questions (2.0%)
  • COMPUTER NETWORK: 13 questions (2.0%)
  • GRAPHICS: 13 questions (2.0%)
  • Q5.: 12 questions (1.8%)
  • PTO: 12 questions (1.8%)
  • COMPUTER GRAPHICS: 12 questions (1.8%)


In [34]:
# Extract real course topics from question data
print("=== ANALYZING 3RD YEAR QUESTION DATA ===\n")

# Get all unique topics and filter out obvious noise
all_topics = third_year_data['topic'].value_counts()

# Filter: keep topics that are meaningful (longer than 4 chars, don't look like codes/noise)
meaningful_topics = all_topics[all_topics.index.str.len() > 4]

# Remove common noise patterns
noise_pattern = r'^\[|^\(|^Q\d|^DS-|^TCS-|^\d+X|^-|-$|^[A-Z]\)|PTO|^10X|EXAMINATION|SEMESTER'
meaningful_topics = meaningful_topics[~meaningful_topics.index.str.contains(noise_pattern, regex=True, case=False)]

print("Top 20 Important Topics for 3rd Year Students:\n")
for idx, (topic, count) in enumerate(meaningful_topics.head(20).items(), 1):
    percentage = (count / len(third_year_data)) * 100
    bar = "█" * int(percentage / 0.5)
    print(f"{idx:2d}. {topic:<50} | {count:3d} Qs ({percentage:5.1f}%) {bar}")

# Save clean topics
important_topics_clean = pd.DataFrame({
    'Rank': range(1, 21),
    'Topic': meaningful_topics.head(20).index.tolist(),
    'Question_Count': meaningful_topics.head(20).values.tolist(),
    'Percentage': ((meaningful_topics.head(20).values / len(third_year_data)) * 100).round(1)
})
important_topics_clean.to_csv('data/important_topics_3rd_year.csv', index=False)
print(f"\n✅ Saved to: data/important_topics_3rd_year.csv")

=== ANALYZING 3RD YEAR QUESTION DATA ===

Top 20 Important Topics for 3rd Year Students:

 1. DATA ANALYTICS                                     |  34 Qs (  5.2%) ██████████
 2. ARTIFICIAL INTELLIGENCE                            |  24 Qs (  3.7%) ███████
 3. COMPUTER NETWORKS                                  |  22 Qs (  3.4%) ██████
 4. ANALYSIS OF ALGORITHMS                             |  13 Qs (  2.0%) ███
 5. COMPUTER NETWORK                                   |  13 Qs (  2.0%) ███
 6. GRAPHICS                                           |  13 Qs (  2.0%) ███
 7. COMPUTER GRAPHICS                                  |  12 Qs (  1.8%) ███
 8. CST021_005 |                                       |  11 Qs (  1.7%) ███
 9. Section B                                          |  10 Qs (  1.5%) ███
10. 203-30T                                            |  10 Qs (  1.5%) ███
11. DATABASE MANAGEMENT SYSTEM                         |  10 Qs (  1.5%) ███
12. ST021_005 | 103.170.7                        

In [35]:
# Generate sample predictions for key topics
print("\n=== SAMPLE PREDICTIONS FOR KEY 3RD YEAR TOPICS ===\n")

key_topics = ['DATA ANALYTICS', 'ARTIFICIAL INTELLIGENCE', 'COMPUTER NETWORKS']
for key_topic in key_topics:
    topic_data = third_year_data[third_year_data['topic'].str.contains(key_topic, case=False, na=False)]
    if len(topic_data) > 0:
        sample = topic_data.iloc[0]
        prediction = predict(sample['topic'], 'Year 3', sample['normalized_question'])
        print(f"📚 Topic: {key_topic}")
        print(f"   Question: {sample['question_text'][:70]}...")
        print(f"   Predicted: {prediction}\n")

print("\n" + "="*80)
print("✅ ANALYSIS COMPLETE!")
print("="*80)


=== SAMPLE PREDICTIONS FOR KEY 3RD YEAR TOPICS ===

📚 Topic: DATA ANALYTICS
   Question: a) Define HDFS?...
   Predicted: 

📚 Topic: ARTIFICIAL INTELLIGENCE
   Question: between informed and uninformed search strategies in AI?...
   Predicted: 

📚 Topic: COMPUTER NETWORKS
   Question: a) What are two reasons for using layered protocols?...
   Predicted: 


✅ ANALYSIS COMPLETE!


## 8. Save Model

In [21]:
model.save_pretrained('saved_model/')
tokenizer.save_pretrained('saved_model/')
print('Model saved.')

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.94it/s]

Model saved.
